# 02 — Customer segmentation (RFM + k-means)

**Questions**
- Can we split registered customers into useful groups with Recency / Frequency / Monetary?
- Does silhouette prefer k=3..6, or is k=4 close enough to label cleanly?

Registered only (null / guest CustomerID rows excluded).


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

ROOT = Path('..')
GOLD = ROOT / 'data' / 'gold'
OUT = Path('outputs')
REPORTS = ROOT / 'reports'
OUT.mkdir(parents=True, exist_ok=True)
REPORTS.mkdir(parents=True, exist_ok=True)

feat = pd.read_csv(GOLD / 'ml_customer_features.csv')
feat = feat[feat['CustomerKey'] > 0].copy()
cols = ['RecencyDays', 'Frequency', 'Monetary']
X = feat[cols].copy()
X['Monetary_log'] = np.log1p(X['Monetary'])
X['Frequency_log'] = np.log1p(X['Frequency'])
use = ['RecencyDays', 'Frequency_log', 'Monetary_log']
scaler = StandardScaler()
Xs = scaler.fit_transform(X[use])
print('customers', len(feat))


In [ ]:
sample_idx = np.random.RandomState(42).choice(len(Xs), size=min(4000, len(Xs)), replace=False)
silhouettes = {}
for k in range(3, 7):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(Xs[sample_idx])
    silhouettes[k] = float(silhouette_score(Xs[sample_idx], labels))
best_k = max(silhouettes, key=silhouettes.get)
if silhouettes.get(4, 0) >= silhouettes[best_k] - 0.02:
    best_k = 4

km = KMeans(n_clusters=best_k, random_state=42, n_init=10)
feat['Cluster'] = km.fit_predict(Xs)
full_sil = float(silhouette_score(Xs, feat['Cluster']))
print(f'fitted k={best_k}  silhouette={full_sil:.3f}')
print('sweep', {k: round(v, 4) for k, v in silhouettes.items()})


In [ ]:
summary = feat.groupby('Cluster').agg(
    Customers=('CustomerKey', 'count'),
    RecencyDays=('RecencyDays', 'median'),
    Frequency=('Frequency', 'median'),
    Monetary=('Monetary', 'median'),
    Revenue=('Monetary', 'sum'),
    RepeatRate=('IsRepeat', 'mean'),
    Churn90Rate=('Churned90', 'mean'),
).reset_index()

def name_row(r):
    if r['RecencyDays'] <= summary['RecencyDays'].median() and r['Monetary'] >= summary['Monetary'].median() and r['Frequency'] >= summary['Frequency'].median():
        return 'Champions'
    if r['RecencyDays'] > summary['RecencyDays'].quantile(0.6) and r['Frequency'] <= summary['Frequency'].median():
        return 'At Risk'
    if r['Frequency'] <= summary['Frequency'].quantile(0.4) and r['RecencyDays'] <= summary['RecencyDays'].median():
        return 'New / Promising'
    if r['Monetary'] >= summary['Monetary'].median():
        return 'Loyal Spenders'
    return 'Need Attention'

summary['Segment'] = summary.apply(name_row, axis=1)
seen, names = {}, []
for s in summary['Segment']:
    if s not in seen:
        seen[s] = 0
        names.append(s)
    else:
        seen[s] += 1
        names.append(f'{s} ({seen[s]+1})')
summary['Segment'] = names
seg_map = dict(zip(summary['Cluster'], summary['Segment']))
feat['Segment'] = feat['Cluster'].map(seg_map)
feat.to_csv(GOLD / 'customer_segments.csv', index=False)
summary.to_csv(OUT / 'segment_summary.csv', index=False)
summary


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
colors = ['#F2C811', '#60A5FA', '#2DD4BF', '#F87171', '#A78BFA', '#FB923C']
ax = axes[0]
for i, c in enumerate(sorted(feat['Cluster'].unique())):
    sub = feat[feat['Cluster'] == c]
    ax.scatter(sub['RecencyDays'], np.log1p(sub['Monetary']), s=12, alpha=0.45, c=colors[i % len(colors)], label=seg_map[c])
ax.set_xlabel('Recency (days)')
ax.set_ylabel('log(1+Monetary)')
ax.set_title('RFM clusters')
ax.legend(fontsize=8)

ax2 = axes[1]
order = summary.sort_values('Revenue', ascending=True)
ax2.barh(order['Segment'], order['Revenue'] / 1e6, color='#F2C811')
ax2.set_xlabel('Revenue (£M)')
ax2.set_title('Segment revenue contribution')
fig.tight_layout()
fig.savefig(OUT / 'customer_segments.png', dpi=140)
plt.show()

metrics = {
    'model': 'KMeans RFM', 'features': use, 'k': int(best_k),
    'silhouette_k_sweep': silhouettes, 'silhouette_full': round(full_sil, 4),
    'n_customers': int(len(feat)), 'segments': summary.to_dict(orient='records'),
}
(REPORTS / 'segmentation_metrics.json').write_text(json.dumps(metrics, indent=2, default=str))
